<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/6_2_Kernel_PCA_and_Kernel_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part VI. Kernel Methods

## Kernel Trick, Mercer's Theorem, and Kernel Methods

## Kernel PCA and Kernel Logistic Regression

## Gaussian Processes

### Topic 2. Ядровый PCA и ядровая логистическая регрессия

#### 1. PCA в пространстве признаков: центрирование и собственные векторы

Ядровой трюк, изученный нами в предыдущих разделах, не ограничивается задачами обучения с учителем. Его можно с равным успехом применить и к методам снижения размерности и выделения признаков без учителя. Классический пример — метод главных компонент (PCA). Перенесённый в спрямляющее пространство $\mathcal{H}$, он превращается в **ядровый PCA** (Kernel PCA) — мощный инструмент нелинейного анализа данных, позволяющий находить направления максимальной дисперсии в бесконечномерных признаковых пространствах, не вычисляя явно отображение $\phi$. В этом разделе мы выведем алгоритм Kernel PCA, начиная с обычного линейного PCA и последовательно заменяя все операции, зависящие от явных координат, их ядровыми аналогами.

---

##### 1.1. Линейный PCA как проекция на главные компоненты

Напомним кратко классический PCA. Пусть дана матрица центрированных данных $\mathbf{X} \in \mathbb{R}^{n \times D}$, строки которой — наблюдения $\mathbf{x}_i^\top \in \mathbb{R}^D$, причём $\sum_{i=1}^n \mathbf{x}_i = \mathbf{0}$. Выборочная ковариационная матрица равна $\mathbf{C} = \frac{1}{n} \mathbf{X}^\top \mathbf{X} \in \mathbb{R}^{D \times D}$. Главные компоненты — это ортонормированные собственные векторы $\mathbf{u}_k$ матрицы $\mathbf{C}$, соответствующие наибольшим собственным числам $\lambda_k$:

$$
\mathbf{C} \mathbf{u}_k = \lambda_k \mathbf{u}_k, \qquad \mathbf{u}_k^\top \mathbf{u}_l = \delta_{kl}.
$$

Проекция точки $\mathbf{x}_i$ на $k$-ю главную компоненту есть скаляр $\langle \mathbf{x}_i, \mathbf{u}_k \rangle$.

Альтернативно, можно рассмотреть матрицу Грама $\mathbf{G} = \mathbf{X} \mathbf{X}^\top \in \mathbb{R}^{n \times n}$, элементы которой суть попарные скалярные произведения $\mathbf{G}_{ij} = \mathbf{x}_i^\top \mathbf{x}_j$. Нетрудно показать, что если $\mathbf{v}_k$ — собственный вектор $\mathbf{G}$ с собственным числом $\mu_k$, то $\mathbf{u}_k = \frac{1}{\sqrt{\mu_k}} \mathbf{X}^\top \mathbf{v}_k$ является нормированным собственным вектором ковариационной матрицы $\mathbf{C}$, а $\mu_k = n \lambda_k$. Эта двойственность — ключ к ядровому обобщению: она позволяет выразить всё через скалярные произведения, избегая явной работы в $D$-мерном пространстве признаков.

---

##### 1.2. Центрирование в пространстве признаков

Перенесём эти идеи в RKHS. Пусть $\phi : \mathcal{X} \to \mathcal{H}$ — отображение в спрямляющее пространство, и $\Phi = [\phi(\mathbf{x}_1), \dots, \phi(\mathbf{x}_n)]$ — (неявная) матрица «признаков» размера $\dim(\mathcal{H}) \times n$. Для выполнения PCA в $\mathcal{H}$ данные должны быть центрированы — среднее отображённых векторов должно равняться нулю. Однако мы не можем вычислить среднее $\frac{1}{n} \sum_i \phi(\mathbf{x}_i)$ явно, если $\mathcal{H}$ бесконечномерно. Но можно работать непосредственно с центрированной ядерной матрицей.

Центрированные образы имеют вид $\tilde{\phi}(\mathbf{x}_i) = \phi(\mathbf{x}_i) - \frac{1}{n} \sum_{j=1}^n \phi(\mathbf{x}_j)$. В матричной форме:

$$
\tilde{\Phi} = \Phi - \frac{1}{n} \Phi \mathbf{1} \mathbf{1}^\top = \Phi \mathbf{H},
$$

где $\mathbf{H} = \mathbf{I} - \frac{1}{n} \mathbf{1} \mathbf{1}^\top$ — центрирующая матрица ($\mathbf{1}$ — вектор из $n$ единиц). Матрица $\mathbf{H}$ симметрична, идемпотентна ($\mathbf{H}^2 = \mathbf{H}$) и обнуляет среднее: $\mathbf{H} \mathbf{1} = \mathbf{0}$.

Матрица Грама для центрированных образов, т.е. центрированная ядерная матрица $\tilde{\mathbf{K}}$, выражается через исходную $\mathbf{K}_{ij} = \langle \phi(\mathbf{x}_i), \phi(\mathbf{x}_j) \rangle_{\mathcal{H}}$ следующим образом:

$$
\tilde{\mathbf{K}} = \tilde{\Phi}^\top \tilde{\Phi} = \mathbf{H} \Phi^\top \Phi \mathbf{H} = \mathbf{H} \mathbf{K} \mathbf{H}. \tag{1.1}
$$

Раскрывая, получаем явную формулу, не требующую знания $\phi$:

$$
\tilde{\mathbf{K}} = \mathbf{K} - \frac{1}{n} \mathbf{K} \mathbf{1} \mathbf{1}^\top - \frac{1}{n} \mathbf{1} \mathbf{1}^\top \mathbf{K} + \frac{1}{n^2} (\mathbf{1}^\top \mathbf{K} \mathbf{1}) \mathbf{1} \mathbf{1}^\top. \tag{1.2}
$$

Таким образом, центрирование в $\mathcal{H}$ сводится к чисто алгебраической операции над матрицей $\mathbf{K}$, вычислимой за $O(n^2)$.

---

##### 1.3. Алгоритм Kernel PCA

Имея центрированную ядерную матрицу $\tilde{\mathbf{K}}$, мы можем выполнить PCA в $\mathcal{H}$, опираясь исключительно на собственное разложение $\tilde{\mathbf{K}}$. Последовательность шагов такова:

1. **Вычислить матрицу ядра** $\mathbf{K}$ для обучающих данных: $\mathbf{K}_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$.
2. **Центрировать:** $\tilde{\mathbf{K}} = \mathbf{H} \mathbf{K} \mathbf{H}$ по формуле (1.1) или (1.2).
3. **Найти собственные векторы** $\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_n$ матрицы $\tilde{\mathbf{K}}$ и соответствующие собственные числа $\mu_1 \ge \mu_2 \ge \dots \ge \mu_n \ge 0$. Собственные векторы считаем ортонормированными в $\mathbb{R}^n$: $\mathbf{v}_k^\top \mathbf{v}_l = \delta_{kl}$.
4. **Связать с главными компонентами в $\mathcal{H}$.** Вектор главных компонент в $\mathcal{H}$ имеет вид $\mathbf{u}_k = \sum_{i=1}^n (\mathbf{v}_k)_i \tilde{\phi}(\mathbf{x}_i)$. Чтобы $\mathbf{u}_k$ был ортонормирован в $\mathcal{H}$, коэффициент нормировки должен быть $1/\sqrt{\mu_k}$. Действительно,
   $$
   \|\mathbf{u}_k\|_{\mathcal{H}}^2 = \sum_{i,j} (\mathbf{v}_k)_i (\mathbf{v}_k)_j \tilde{K}_{ij} = \mathbf{v}_k^\top \tilde{\mathbf{K}} \mathbf{v}_k = \mu_k \|\mathbf{v}_k\|^2 = \mu_k,
   $$
   поэтому нормированный собственный вектор равен $\mathbf{u}_k = \frac{1}{\sqrt{\mu_k}} \sum_{i=1}^n (\mathbf{v}_k)_i \tilde{\phi}(\mathbf{x}_i)$.

Таким образом, мы получаем ортонормированный базис главных направлений в $\mathcal{H}$, не покидая пространства коэффициентов.

---

##### 1.4. Проекция новой точки

Одно из важнейших применений PCA — понижение размерности новых наблюдений. Для ядрового PCA проекция тестовой точки $\mathbf{x}_*$ на $k$-ю главную компоненту даётся скалярным произведением

$$
z_k(\mathbf{x}_*) = \langle \tilde{\phi}(\mathbf{x}_*), \mathbf{u}_k \rangle_{\mathcal{H}} = \frac{1}{\sqrt{\mu_k}} \sum_{i=1}^n (\mathbf{v}_k)_i \langle \tilde{\phi}(\mathbf{x}_*), \tilde{\phi}(\mathbf{x}_i) \rangle_{\mathcal{H}}.
$$

Осталось выразить центрированное скалярное произведение $\langle \tilde{\phi}(\mathbf{x}_*), \tilde{\phi}(\mathbf{x}_i) \rangle_{\mathcal{H}}$ через исходное ядро. Используя определение центрирования, получаем

$$
\langle \tilde{\phi}(\mathbf{x}_*), \tilde{\phi}(\mathbf{x}_i) \rangle_{\mathcal{H}} = K(\mathbf{x}_*, \mathbf{x}_i) - \frac{1}{n} \sum_{j=1}^n K(\mathbf{x}_*, \mathbf{x}_j) - \frac{1}{n} \sum_{j=1}^n K(\mathbf{x}_j, \mathbf{x}_i) + \frac{1}{n^2} \sum_{j,l} K(\mathbf{x}_j, \mathbf{x}_l).
$$

В векторной форме: определим столбец $\mathbf{k}_* = [K(\mathbf{x}_*, \mathbf{x}_1), \dots, K(\mathbf{x}_*, \mathbf{x}_n)]^\top$. Тогда центрированный вектор $\tilde{\mathbf{k}}_*$ получается как

$$
\tilde{\mathbf{k}}_* = \mathbf{k}_* - \frac{1}{n} \mathbf{K} \mathbf{1} - \frac{1}{n} \mathbf{1} \mathbf{1}^\top \mathbf{k}_* + \frac{1}{n^2} (\mathbf{1}^\top \mathbf{K} \mathbf{1}) \mathbf{1}. \tag{1.3}
$$

Проекция на $k$-ю компоненту:

$$
z_k(\mathbf{x}_*) = \frac{1}{\sqrt{\mu_k}} \, \tilde{\mathbf{k}}_*^\top \mathbf{v}_k. \tag{1.4}
$$

Обратим внимание: поскольку ранг $\tilde{\mathbf{K}}$ не превышает $n$, число ненулевых собственных значений не больше $n$, а значит, мы можем извлечь не более $n$ главных компонент.

---

##### 1.5. Углублённый взгляд: связь Kernel PCA с многомерным шкалированием (MDS)

> **Для читателя с математической подготовкой.** Kernel PCA тесно связан с классическим многомерным шкалированием (MDS). Напомним, что MDS по матрице квадратов попарных расстояний $D_{ij} = \|\mathbf{x}_i - \mathbf{x}_j\|^2$ строит координаты точек, сохраняющие эти расстояния. Если выполнить MDS с ядром $K_{\text{cent}}(\mathbf{x}_i, \mathbf{x}_j) = -\frac12 D_{ij}$ и применить двойное центрирование $\mathbf{H} \mathbf{K} \mathbf{H}$, то полученные координаты в точности совпадают с проекциями на главные компоненты Kernel PCA с ядром, индуцированным расстоянием.
>
> В частности, если взять линейное ядро $K(\mathbf{x}, \mathbf{y}) = \mathbf{x}^\top \mathbf{y}$ и центрированные данные, то $\tilde{\mathbf{K}} = \mathbf{X} \mathbf{X}^\top$. Собственные векторы $\tilde{\mathbf{K}}$ с точностью до масштаба дадут проекции на классические главные компоненты. Таким образом, Kernel PCA включает классический PCA как частный случай.
>
> Более того, MDS можно рассматривать как Kernel PCA с ядром, неявно заданным через расстояние. Для произвольной симметричной матрицы различий можно построить ядро, если матрица двойного центрирования $-\frac12 \mathbf{H} \mathbf{D} \mathbf{H}$ окажется неотрицательно определённой. Эта глубокая связь объединяет PCA, MDS и спектральное вложение и позволяет переносить геометрическую интуицию из одного метода в другой.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Зачем нужно центрировать данные в пространстве признаков при выполнении Kernel PCA? Что произойдёт, если этого не сделать?
2. Выпишите формулу центрированной ядерной матрицы $\tilde{\mathbf{K}}$ через исходную матрицу $\mathbf{K}$ и объясните каждое слагаемое.
3. Что представляет собой матрица $\mathbf{H} = \mathbf{I} - \frac{1}{n} \mathbf{1} \mathbf{1}^\top$? Какими свойствами она обладает?
4. Как вычислить проекцию новой точки $\mathbf{x}_*$ на $k$-ю главную компоненту в пространстве $\mathcal{H}$, не зная отображения $\phi$?
5. Сколько главных компонент можно извлечь с помощью Kernel PCA на $n$ точках? Почему?

**Для профессионалов:**

1. Выведите формулу центрирования $\tilde{\mathbf{K}} = \mathbf{H} \mathbf{K} \mathbf{H}$ и покажите, что она эквивалентна выражению (1.2). Объясните роль свойства идемпотентности $\mathbf{H}$.
2. Докажите, что собственные векторы матрицы $\tilde{\mathbf{K}}$ связаны с собственными векторами ковариационной матрицы в $\mathcal{H}$: если $\tilde{\mathbf{K}} \mathbf{v}_k = \mu_k \mathbf{v}_k$, то $\mathbf{u}_k = \frac{1}{\sqrt{\mu_k}} \sum_i (\mathbf{v}_k)_i \tilde{\phi}(\mathbf{x}_i)$ — собственный вектор ковариационной матрицы с тем же собственным числом $\mu_k/n$.
3. Объясните, почему число ненулевых собственных значений $\tilde{\mathbf{K}}$ не превышает $n$. Как это связано с размерностью пространства $\mathcal{H}$?
4. Сравните Kernel PCA с методом главных кривых (principal curves). В чём принципиальное различие в определении нелинейных направлений максимальной вариации?

### 2. Реализация Kernel PCA на Python и визуализация

В предыдущем разделе мы вывели математические основы ядрового PCA: центрирование в пространстве признаков, собственное разложение центрированной ядерной матрицы и проекцию новых точек. Теперь мы превратим эти формулы в вычислительный инструмент, написав собственную реализацию Kernel PCA на Python и сравнив её с промышленной библиотекой scikit-learn. В качестве тестового полигона выступит классическое нелинейное многообразие — «швейцарский рулет» (Swiss roll), на котором мы наглядно увидим, как ядровый PCA «разворачивает» скрученные данные в плоскость.

---

#### 2.1. Структура класса KernelPCA

Реализация естественным образом разбивается на три этапа: обучение (`fit`), преобразование обучающих данных (`transform`) и преобразование новых тестовых данных (с помощью того же метода `transform`). Проектируя класс, мы закладываем в него следующие параметры, соответствующие стандартной функциональности:

- `kernel` — тип ядра: `'linear'`, `'poly'`, `'rbf'`, `'sigmoid'` и т.д.
- `gamma`, `degree`, `coef0` — параметры выбранного ядра.
- `n_components` — число главных компонент, которые необходимо сохранить.
- Дополнительно можно предусмотреть флаг `center=True` (центрировать ли данные в $\mathcal{H}$; почти всегда нужно).

Опишем ключевые методы.

**Метод `fit(X)`** (обучение по матрице `X` формы `(n_samples, n_features)`):

1. Вычислить ядерную матрицу `K` размера $n \times n$: `K[i,j] = kernel(X[i], X[j])`. Для эффективности используют векторизованные операции: для RBF-ядра `K = exp(-gamma * pairwise_sq_dists)`.
2. Выполнить центрирование: `K_centered = H @ K @ H`, где `H = np.eye(n) - 1/n * np.ones((n,n))`. На практике можно избежать плотного умножения, но для учебных целей допустимо прямое.
3. Найти собственные значения и собственные векторы `K_centered` с помощью `np.linalg.eigh` (для симметричных матриц). Собственные значения по возрастанию; мы переворачиваем массив, чтобы получить их по убыванию, и берём первые `n_components` наибольших.
4. Сохранить собственные векторы `alphas_` (`(n_components, n)` — каждая строка соответствует одному собственному вектору) и собственные значения `lambdas_` (одномерный массив длины `n_components`).

**Метод `transform(X)`** (проекция данных `X` на главные компоненты, может совпадать с обучающими или быть новыми):

1. Вычислить ядерную матрицу между `X` и исходными обучающими точками `X_train`: `K_test` размера `(n_new, n_train)`.
2. Центрировать тестовую ядерную матрицу, используя статистики, вычисленные на этапе обучения. Формула центрирования (1.3) в матричном виде для всего тестового набора:
   $$
   \tilde{\mathbf{K}}_{\text{test}} = \mathbf{K}_{\text{test}} - \frac{1}{n} \mathbf{K}_{\text{test}} \mathbf{1}\mathbf{1}^\top - \frac{1}{n} \mathbf{1}\mathbf{1}^\top \mathbf{K} + \frac{1}{n^2} (\mathbf{1}^\top \mathbf{K} \mathbf{1}) \mathbf{1}\mathbf{1}^\top,
   $$
   где $\mathbf{K}$ — исходная (нецентрированная) матрица обучающих данных. Здесь $\mathbf{K}_{\text{test}}$ умножается на матрицу из единиц соответствующего размера.
3. Спроецировать центрированные тестовые ядерные векторы на собственные векторы, нормированные на квадратный корень из собственных чисел:
   $$
   \mathbf{Z} = \tilde{\mathbf{K}}_{\text{test}} \cdot \mathbf{A} \cdot \operatorname{diag}(1/\sqrt{\lambda_1}, \dots, 1/\sqrt{\lambda_d}),
   $$
   где $\mathbf{A}$ — матрица, строки которой — собственные векторы $\mathbf{v}_k^\top$. Результат $\mathbf{Z}$ — матрица проекций размера `(n_new, n_components)`.

При реализации важно обратить внимание на численную устойчивость: отсекать очень маленькие собственные числа (меньше $10^{-12}$) во избежание деления на ноль.

---

#### 2.2. Визуализация: «швейцарский рулет»

Чтобы продемонстрировать мощь Kernel PCA, сгенерируем трёхмерное множество точек, лежащих на свёрнутой поверхности — Swiss roll. Классический способ: параметризуем точки углом $t \in [3\pi/2, 9\pi/2]$ и высотой $z$, затем скручиваем в рулет. Каждая точка получает цвет, соответствующий её положению на развёртке (например, по углу $t$), что создаёт гладкий цветовой градиент вдоль многообразия.

1. **Исходные данные в 3D** визуализируются точечным облаком, окрашенным согласно параметру $t$. Хорошо видно, что точки лежат на двумерной поверхности, свёрнутой в трёхмерном пространстве.
2. **Линейный PCA**, применённый к трёхмерным координатам, проецирует облако на плоскость, но не способен развернуть рулет: проекция оказывается «сплющенной», сохраняя глобальную скрученность и перемешивая цвета.
3. **Kernel PCA с RBF-ядром** радикально меняет картину. При правильно подобранном $\gamma$ первые две главные компоненты «разворачивают» многообразие в плоскость, и цветовой градиент становится плавным, отражая истинную внутреннюю параметризацию. Это наглядно демонстрирует, что ядровый PCA находит нелинейные направления максимальной вариации, соответствующие скрытой структуре данных.

Сравнение с `sklearn.decomposition.KernelPCA` подтверждает корректность нашей реализации: проекции и объяснённая дисперсия (собственные числа) совпадают с точностью до знака компонент.

---

#### 2.3. Влияние параметра $\gamma$ RBF-ядра

Параметр $\gamma$ RBF-ядра определяет характерный масштаб близости. Варьирование $\gamma$ позволяет исследовать структуру данных на разных уровнях детализации.

- При очень **малом $\gamma$** (широкое ядро) все точки сильно связаны, матрица $\mathbf{K}$ близка к матрице из единиц. Центрированная ядерная матрица вырождается, и главные компоненты отражают глобальные, почти линейные тренды. Swiss roll разворачивается лишь частично: компоненты улавливают самые крупномасштабные вариации.
- При **умеренном $\gamma$** (сравнимом с обратным квадратом характерного расстояния между точками) Kernel PCA успешно восстанавливает внутреннюю геометрию многообразия. Проекция даёт чёткий, плавно окрашенный прямоугольник.
- При очень **большом $\gamma$** (узкое ядро) ядерная матрица становится почти диагональной: каждая точка «чувствует» только саму себя. В этом случае Kernel PCA вырождается: собственные векторы соответствуют изолированным точкам, и проекция распадается на отдельные кластеры (или даже отдельные точки), что ведёт к переобучению и потере информации о глобальной структуре.

Визуализация проекций при нескольких значениях $\gamma$ на одной фигуре (например, сеткой $2 \times 3$) наглядно демонстрирует этот переход от глобального к локальному.

---

#### 2.4. Вычислительная сложность и проблема масштабирования

Полная реализация Kernel PCA требует $O(n^2)$ памяти для хранения ядерной матрицы и $O(n^3)$ времени для её собственного разложения. При $n \approx 10^4$ это ещё приемлемо (матрица занимает около 800 МБ для `float64`), но для $n = 10^5$ и более прямое решение становится нереализуемым.

Для масштабирования применяют приближённые методы:

- **Аппроксимация Найстрёма (Nyström):** выбирается $m \ll n$ опорных точек, строится приближение низкого ранга $\tilde{\mathbf{K}} \approx \mathbf{K}_{nm} \mathbf{K}_{mm}^{-1} \mathbf{K}_{nm}^\top$. Собственное разложение проводится на матрице $m \times m$, а проекции восстанавливаются через матрицу перекрёстных ядер. Сложность снижается до $O(n m^2 + m^3)$.
- **Landmark Kernel PCA:** аналог предыдущего, но вместо полного разложения ищутся собственные векторы приближённой матрицы.
- **Случайные признаки (Random Fourier Features)** для стационарных ядер позволяют заменить бесконечномерное отображение конечномерным и выполнить обычный линейный PCA в пространстве размерности $D \ll n$, что даёт сложность $O(n D^2 + D^3)$.

В учебной реализации можно предусмотреть опциональный параметр `n_landmarks`, при указании которого используется Найстрём-приближение.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Как центрировать тестовую точку при проекции в Kernel PCA? Почему нельзя просто вычислить $K(\mathbf{x}_*, \mathbf{x}_i)$ и подставить в формулу без центрирования?
2. Почему собственные векторы центрированной ядерной матрицы задают коэффициенты разложения, а не сами главные компоненты? Как получить направления в $\mathcal{H}$?
3. Как выбрать число главных компонент в Kernel PCA? Какие критерии можно использовать?
4. Зачем нужна нормировка собственных векторов на $1/\sqrt{\lambda_k}$? Что произойдёт, если её опустить?

**Для профессионалов:**

1. Реализуйте формулу центрирования тестовой ядерной матрицы без вычисления полной матрицы $\mathbf{K}_{\text{test}} \mathbf{1}\mathbf{1}^\top$ (используйте суммирование по столбцам). Сравните по времени с наивной версией.
2. Объясните, как интерпретировать первые две компоненты Kernel PCA на Swiss roll как нелинейные координаты, параметризующие многообразие. Как эта параметризация соотносится с истинными порождающими параметрами?
3. Сравните время работы вашей реализации и `sklearn.decomposition.KernelPCA` при разных $n$. Какие оптимизации использует библиотечная версия?
4. Добавьте в свой класс поддержку предвычисленного ядра: если на вход `fit` подаётся готовая матрица $\mathbf{K}$, пропускать шаг её вычисления. В каких сценариях это полезно?

### 3. Денузинг и реконструкция в Kernel PCA

Ядровый PCA не только вскрывает нелинейную структуру данных, но и может быть использован для их очистки от шума — **денузинга**. В линейном PCA восстановление очищенной точки по её проекции на главные компоненты тривиально: это линейная комбинация базисных векторов. В RKHS прямое восстановление $\hat{\phi}$ также является линейной операцией, однако ему соответствует точка в исходном пространстве $\mathcal{X}$ только тогда, когда образ $\hat{\phi}$ лежит в образе отображения $\phi(\mathcal{X})$. Поиск такой точки, называемый **задачей предобраза** (pre-image problem), составляет центральную трудность и, одновременно, изюминку нелинейного денузинга с помощью Kernel PCA.

---

#### 3.1. Идея денузинга: от линейного PCA к нелинейному

В линейном PCA данные $\mathbf{x} \in \mathbb{R}^D$ проецируются на $k$-мерное подпространство, натянутое на первые $k$ главных компонент $\mathbf{u}_1, \dots, \mathbf{u}_k$. Очищенная (денуизированная) версия точки $\mathbf{x}$ есть её проекция на это подпространство:

$$
\hat{\mathbf{x}} = \sum_{j=1}^k \langle \mathbf{x}, \mathbf{u}_j \rangle \mathbf{u}_j = \mathbf{U}_k \mathbf{U}_k^\top \mathbf{x},
$$

где столбцы матрицы $\mathbf{U}_k$ — векторы $\mathbf{u}_j$. Эта операция уменьшает шум, отбрасывая компоненты, соответствующие малым собственным числам.

В ядровом PCA мы работаем в спрямляющем пространстве $\mathcal{H}$. Для точки $\mathbf{x}$ её образ $\phi(\mathbf{x})$ проецируется на подпространство $\mathcal{H}_k$, натянутое на первые $k$ главных направлений $\mathbf{u}_1, \dots, \mathbf{u}_k \in \mathcal{H}$. Проекция в $\mathcal{H}$ имеет вид

$$
\hat{\phi}(\mathbf{x}) = \sum_{j=1}^k \langle \phi(\mathbf{x}), \mathbf{u}_j \rangle_{\mathcal{H}} \mathbf{u}_j.
$$

Используя представление $\mathbf{u}_j = \sum_{i=1}^n (\mathbf{v}_j)_i \tilde{\phi}(\mathbf{x}_i)$, где $\mathbf{v}_j$ — $j$-й собственный вектор центрированной ядерной матрицы $\tilde{\mathbf{K}}$ с собственным числом $\mu_j$, а $\tilde{\phi}(\mathbf{x}_i)$ — центрированные образы, мы можем записать $\hat{\phi}(\mathbf{x})$ как линейную комбинацию центрированных образов обучающих точек:

$$
\hat{\phi}(\mathbf{x}) = \sum_{i=1}^n \alpha_i \tilde{\phi}(\mathbf{x}_i), \qquad \alpha_i = \sum_{j=1}^k \frac{(\mathbf{v}_j)_i}{\sqrt{\mu_j}} \langle \tilde{\phi}(\mathbf{x}), \mathbf{u}_j \rangle_{\mathcal{H}} = \sum_{j=1}^k \frac{(\mathbf{v}_j)_i}{\mu_j} (\tilde{\mathbf{k}}_*^\top \mathbf{v}_j),
$$

где $\tilde{\mathbf{k}}_*$ — центрированный вектор ядер тестовой точки. Таким образом, проекция $\hat{\phi}$ в $\mathcal{H}$ вычисляется без знания $\phi$. Однако $\hat{\phi}$ — это элемент RKHS, и чтобы получить точку в исходном пространстве $\mathcal{X}$, нужно найти $\mathbf{z} \in \mathcal{X}$ такой, что $\phi(\mathbf{z}) \approx \hat{\phi}$. Это и есть **задача предобраза**.

---

#### 3.2. Поиск предобраза для RBF-ядра

Рассмотрим часто используемое RBF-ядро $K(\mathbf{x}, \mathbf{x}') = \exp(-\gamma \|\mathbf{x} - \mathbf{x}'\|^2)$. Задача поиска предобраза ставится как минимизация расстояния в $\mathcal{H}$ между образом искомой точки $\phi(\mathbf{z})$ и заданной линейной комбинацией $\boldsymbol{\psi} = \sum_{i=1}^n \alpha_i \tilde{\phi}(\mathbf{x}_i)$:

$$
\mathbf{z}^* = \arg\min_{\mathbf{z} \in \mathcal{X}} \|\phi(\mathbf{z}) - \boldsymbol{\psi}\|_{\mathcal{H}}^2.
$$

Раскроем квадрат нормы, используя тождество $\|\phi(\mathbf{z}) - \boldsymbol{\psi}\|^2 = \|\phi(\mathbf{z})\|^2 - 2\langle \phi(\mathbf{z}), \boldsymbol{\psi} \rangle + \|\boldsymbol{\psi}\|^2$. Для RBF-ядра $\|\phi(\mathbf{z})\|^2 = K(\mathbf{z}, \mathbf{z}) = 1$ постоянно, поэтому задача эквивалентна максимизации скалярного произведения

$$
J(\mathbf{z}) = \langle \phi(\mathbf{z}), \boldsymbol{\psi} \rangle_{\mathcal{H}} = \sum_{i=1}^n \alpha_i \langle \phi(\mathbf{z}), \tilde{\phi}(\mathbf{x}_i) \rangle_{\mathcal{H}}.
$$

Центрированное скалярное произведение $\langle \phi(\mathbf{z}), \tilde{\phi}(\mathbf{x}_i) \rangle_{\mathcal{H}}$ выражается через ядро аналогично (1.3):

$$
\langle \phi(\mathbf{z}), \tilde{\phi}(\mathbf{x}_i) \rangle_{\mathcal{H}} = K(\mathbf{z}, \mathbf{x}_i) - \frac{1}{n}\sum_{j=1}^n K(\mathbf{z}, \mathbf{x}_j) - \frac{1}{n}\sum_{j=1}^n K(\mathbf{x}_j, \mathbf{x}_i) + \frac{1}{n^2}\sum_{j,l} K(\mathbf{x}_j, \mathbf{x}_l).
$$

Таким образом, $J(\mathbf{z})$ — гладкая функция от $\mathbf{z}$, и для её максимизации можно применить градиентный подъём. Градиент $J(\mathbf{z})$ для RBF-ядра вычисляется аналитически:

$$
\nabla_{\mathbf{z}} J(\mathbf{z}) = 2\gamma \sum_{i=1}^n \alpha_i \bigl[ - K(\mathbf{z}, \mathbf{x}_i) (\mathbf{z} - \mathbf{x}_i) + \frac{1}{n} \sum_{j=1}^n K(\mathbf{z}, \mathbf{x}_j) (\mathbf{z} - \mathbf{x}_j) \bigr].
$$

Запуская итерации $\mathbf{z}^{(t+1)} = \mathbf{z}^{(t)} + \eta \nabla_{\mathbf{z}} J(\mathbf{z}^{(t)})$ из подходящего начального приближения (например, ближайшей обучающей точки или её образа), мы находим предобраз $\mathbf{z}^*$, который и будет очищенной версией исходной точки.

Важно: предобраз определён неоднозначно. Например, точки, симметричные относительно многообразия, могут давать одинаковые проекции. Выбор начального приближения регулирует, в какой «бассейн притяжения» сойдётся оптимизация.

---

#### 3.3. Денузинг на примере двумерных точек

Проиллюстрируем процесс. Сгенерируем точки на одномерном нелинейном многообразии в $\mathbb{R}^2$ — например, на полуокружности или спирали, — и добавим к координатам гауссовский шум. Применим Kernel PCA с RBF-ядром к зашумлённым данным.

- Поскольку истинная размерность данных равна 1, сохраним одну главную компоненту ($k=1$), полагая, что она описывает форму многообразия, а отброшенные компоненты поглощают шум.
- Для каждой зашумлённой точки $\mathbf{x}_i$ вычислим её проекцию в $\mathcal{H}$ на первую главную компоненту: получим коэффициенты $\alpha_i$ и вектор $\boldsymbol{\psi}$.
- Решив задачу предобраза (градиентным подъёмом), получим точку $\mathbf{z}_i^*$ — восстановленную, очищенную от шума версию $\mathbf{x}_i$.

Визуализация показывает: шумные точки, разбросанные вокруг истинной кривой, после денузинга «подтягиваются» к ней, образуя гладкую линию. Это демонстрирует способность Kernel PCA выделять нелинейное многообразие, даже когда оно не задано явно.

---

#### 3.4. Сравнение с линейным PCA

Линейный PCA, применённый к тем же зашумлённым данным на полуокружности, спроецирует их на прямую — направление максимальной дисперсии. Эта прямая не способна отразить кривизну данных; денузинг с одной главной компонентой восстановит точки на прямой, что даст значительно худшее приближение к истинной полуокружности. Kernel PCA выигрывает именно за счёт нелинейного вложения: его главные направления в $\mathcal{H}$ соответствуют нелинейным кривым в $\mathcal{X}$, позволяя моделировать искривлённые многообразия.

Однако это преимущество не бесплатно: денузинг через предобраз требует решения нетривиальной оптимизационной задачи для каждой точки и может быть чувствителен к инициализации и выбору $\gamma$.

---

#### 3.5. Ограничения и особые случаи

- **Неединственность предобраза.** Как упоминалось, функция $J(\mathbf{z})$ может иметь множество локальных максимумов, особенно при больших $k$ или сильно зашумлённых данных. На практике используют несколько случайных стартов и выбирают наилучший результат.
- **Полиномиальное ядро: аналитическое решение.** Для полиномиального ядра $K(\mathbf{x}, \mathbf{x}') = (\gamma \mathbf{x}^\top \mathbf{x}' + r)^d$ предобраз иногда можно найти в замкнутой форме. Например, для однородного квадратичного ядра $K(\mathbf{x}, \mathbf{x}') = (\mathbf{x}^\top \mathbf{x}')^2$ задача сводится к поиску $\mathbf{z}$, минимизирующего невязку между матрицами $\mathbf{z}\mathbf{z}^\top$ и целевой матрицей, что решается через собственное разложение. В общем случае полиномиальные ядра позволяют выразить $\phi$ явно и тем самым свести задачу к линейной или квадратичной оптимизации.
- **Вычислительная сложность.** Каждая итерация градиентного подъёма требует $O(n)$ вычислений ядра, что для больших $n$ может быть затратно. Приближённые методы (Nyström, случайные признаки) помогают ускорить как прямое построение проекции, так и поиск предобраза.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Что такое задача предобраза и почему она возникает при денузинге с Kernel PCA?
2. Почему в RBF-ядре норма $\|\phi(\mathbf{z})\|_{\mathcal{H}}$ постоянна и как это упрощает поиск предобраза?
3. Опишите, как градиентный подъём используется для нахождения предобраза в RBF-ядре.
4. Приведите пример данных, где Kernel PCA денузинг работает значительно лучше линейного PCA. Объясните, почему.
5. Чем отличается операция проекции в $\mathcal{H}$ от восстановления точки в исходном пространстве?

**Для профессионалов:**

1. Выведите выражение для градиента $J(\mathbf{z})$ в случае RBF-ядра и покажите, что оно представимо как взвешенная сумма разностей $\mathbf{z} - \mathbf{x}_i$.
2. Предложите альтернативный метод поиска предобраза, основанный на многомерном шкалировании (MDS). В чём его преимущества и недостатки перед градиентным спуском?
3. Сравните качество денузинга с помощью Kernel PCA для RBF-ядра и полиномиального ядра степени 2 на данных, лежащих на эллипсе. Какое ядро предпочтительнее и почему?
4. Объясните, почему для полиномиального ядра $K(\mathbf{x}, \mathbf{x}') = (\mathbf{x}^\top \mathbf{x}')^d$ предобраз иногда может быть найден аналитически. Какие алгебраические свойства используются?

### 4. Ядровая логистическая регрессия: вывод и регуляризация в RKHS

До сих пор мы применяли ядровой трюк к задачам регрессии (KRR) и обучения без учителя (Kernel PCA). В задачах классификации естественным кандидатом на «ядровизацию» является логистическая регрессия — линейный метод, минимизирующий логарифмическую функцию потерь. Перенесённая в воспроизводящее ядерное гильбертово пространство (RKHS), она превращается в **ядерную логистическую регрессию** (Kernel Logistic Regression, KLR) — нелинейный классификатор, который строит гладкие, вероятностно интерпретируемые решающие границы. В этом разделе мы выведем KLR, исследуем свойства получаемого решения, обсудим метод его обучения (IRLS) и сравним с другим популярным ядровым классификатором — SVM.

---

#### 4.1. Постановка задачи и применение теоремы о представителе

Пусть дана обучающая выборка $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$, $\mathbf{x}_i \in \mathcal{X}$, $y_i \in \{-1, +1\}$. Модель ядерной логистической регрессии задаёт условную вероятность класса через сигмоидную функцию от элемента $f \in \mathcal{H}$, где $\mathcal{H}$ — RKHS с воспроизводящим ядром $K$:

$$
p(y = +1 \mid \mathbf{x}) = \sigma(f(\mathbf{x})), \qquad \sigma(t) = \frac{1}{1 + e^{-t}}.
$$

При $y \in \{-1, +1\}$ логарифмическая функция потерь (кросс-энтропия) для одного примера записывается как $\log(1 + \exp(-y f(\mathbf{x})))$. Для обучения выбираем регуляризованный эмпирический риск:

$$
\min_{f \in \mathcal{H}} \left\{ J(f) = \sum_{i=1}^n \log\!\bigl(1 + \exp(-y_i f(\mathbf{x}_i))\bigr) + \frac{\lambda}{2} \|f\|_{\mathcal{H}}^2 \right\}, \qquad \lambda > 0. \tag{4.1}
$$

Слагаемое $\frac{\lambda}{2}\|f\|_{\mathcal{H}}^2$ играет роль штрафа за сложность функции, контролируя гладкость и предотвращая переобучение.

Согласно теореме о представителе (раздел 3.5), решение задачи (4.1) существует и имеет вид конечной линейной комбинации ядер, центрированных в обучающих точках:

$$
f^*(\mathbf{x}) = \sum_{i=1}^n \alpha_i K(\mathbf{x}, \mathbf{x}_i), \qquad \boldsymbol{\alpha} \in \mathbb{R}^n. \tag{4.2}
$$

Подставляя (4.2) в (4.1) и используя матрицу Грама $\mathbf{K}_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$, получаем конечномерную выпуклую задачу оптимизации относительно вектора коэффициентов $\boldsymbol{\alpha}$:

$$
\min_{\boldsymbol{\alpha} \in \mathbb{R}^n} \left\{ \hat{J}(\boldsymbol{\alpha}) = \sum_{i=1}^n \log\!\bigl(1 + \exp(-y_i (\mathbf{K}\boldsymbol{\alpha})_i)\bigr) + \frac{\lambda}{2} \boldsymbol{\alpha}^\top \mathbf{K} \boldsymbol{\alpha} \right\}. \tag{4.3}
$$

В отличие от задачи KRR, здесь функция потерь не квадратична, поэтому аналитическое решение отсутствует. Однако выпуклость и гладкость позволяют эффективно применять методы ньютоновского типа.

---

#### 4.2. Свойства решения и связь с SVM

Ядерная логистическая регрессия обладает несколькими важными свойствами, которые определяют её место среди ядровых классификаторов.

1.  **Вероятностный выход.** По построению модель оценивает вероятность $p(y = +1 \mid \mathbf{x})$, что полезно во многих приложениях (ранжирование, оценка рисков, калибровка).

2.  **Гладкость и дифференцируемость.** Логарифмическая функция потерь является гладкой и строго выпуклой по $f$, что позволяет использовать методы второго порядка (Ньютон, IRLS) и гарантирует быструю сходимость.

3.  **Отсутствие разреженности.** В отличие от SVM с hinge loss, где многие коэффициенты $\alpha_i$ становятся нулевыми (опорные векторы), в KLR, как правило, все $\alpha_i \neq 0$. Это связано с тем, что логистическая потеря не имеет «плоского» участка, и каждое наблюдение вносит вклад в решение. В результате предсказание требует вычисления ядра со всеми обучающими точками, что делает KLR более затратным на этапе тестирования.

4.  **Связь с максимумом правдоподобия.** При $\lambda \to 0$ KLR стремится к нерегуляризованному максимуму правдоподобия в RKHS. В случае линейного ядра получается классическая логистическая регрессия с $\ell_2$-штрафом (гребневая логистическая регрессия).

5.  **Сравнение с SVM.** SVM минимизирует hinge loss $\max(0, 1 - y f(\mathbf{x}))$, что даёт разреженное решение и нацелено на максимизацию зазора. Hinge loss менее чувствителен к выбросам, но не даёт вероятностей напрямую — для калибровки требуется постобработка (Platt scaling). KLR, напротив, естественно калибрована, но может хуже разделять классы при малых выборках из-за отсутствия встроенного механизма зазора. На практике выбор между ними диктуется требованиями к вероятностной интерпретации и вычислительными ограничениями.

---

#### 4.3. Оптимизация: метод Ньютона и IRLS в RKHS

Для минимизации (4.3) удобно использовать метод Ньютона, который в контексте обобщённых линейных моделей принимает форму итеративно перевзвешиваемого метода наименьших квадратов (Iteratively Reweighted Least Squares, IRLS). Выпишем градиент и гессиан функции $\hat{J}(\boldsymbol{\alpha})$.

Введём обозначение $\mathbf{f} = \mathbf{K}\boldsymbol{\alpha}$ — вектор значений функции на обучающих точках. Тогда логарифмическая потеря для $i$-го примера есть $L_i = \log(1 + e^{-y_i f_i})$. Производная $L_i$ по $f_i$ равна $-\frac{y_i}{1 + e^{y_i f_i}} = -y_i (1 - \sigma(y_i f_i))$. Определим вектор «откликов» и веса:

$$
p_i = \sigma(y_i f_i) = \frac{1}{1 + e^{-y_i f_i}}, \qquad
w_i = p_i (1 - p_i) = \frac{e^{-y_i f_i}}{(1 + e^{-y_i f_i})^2}.
$$

Тогда градиент $\hat{J}$ по $\boldsymbol{\alpha}$:

$$
\nabla_{\boldsymbol{\alpha}} \hat{J} = \mathbf{K} \mathbf{r} + \lambda \mathbf{K} \boldsymbol{\alpha}, \quad \text{где} \quad \mathbf{r}_i = -y_i (1 - p_i).
$$

Гессиан $\hat{J}$:

$$
\nabla^2_{\boldsymbol{\alpha}} \hat{J} = \mathbf{K} \mathbf{W} \mathbf{K} + \lambda \mathbf{K},
$$

где $\mathbf{W} = \operatorname{diag}(w_1, \dots, w_n)$ — диагональная матрица весов, $w_i > 0$. Гессиан положительно определён при $\lambda > 0$ и положительно полуопределённой $\mathbf{K}$, что гарантирует выпуклость.

Шаг метода Ньютона: $\boldsymbol{\alpha}_{\text{new}} = \boldsymbol{\alpha}_{\text{old}} - (\nabla^2 \hat{J})^{-1} \nabla \hat{J}$. Алгебраические преобразования позволяют переписать этот шаг в эквивалентной форме как решение взвешенной линейной системы относительно нового $\boldsymbol{\alpha}$:

$$
(\mathbf{K} \mathbf{W} \mathbf{K} + \lambda \mathbf{K}) \boldsymbol{\alpha}_{\text{new}} = \mathbf{K} \mathbf{W} \mathbf{z},
$$

где $\mathbf{z} = \mathbf{f} + \mathbf{W}^{-1} (\mathbf{y} - \mathbf{p})$ — скорректированный отклик. Эту систему можно эффективно решать, например, через двойственное представление или с использованием разложения Холецкого, если $n$ не слишком велико. Итерации продолжаются до сходимости.

Вычислительная сложность одной итерации — $O(n^3)$ при прямом обращении; на практике применяют итерационные решатели или метод сопряжённых градиентов, особенно для больших $n$. Также возможен вывод в прямой форме (с явным $\phi$) через формулу Вудбери, аналогично KRR.

---

#### 4.4. Углублённый взгляд: калибровка вероятностей в KLR против SVM

> **Для читателя с математической подготовкой.** Одним из ключевых преимуществ KLR является теоретическая гарантия калибровки вероятностей. Минимизация логарифмической функции потерь эквивалентна максимизации правдоподобия Бернулли, поэтому при достаточно богатом RKHS и правильной спецификации модели предсказанные вероятности $\hat{p}(y=1 \mid \mathbf{x}) = \sigma(f^*(\mathbf{x}))$ асимптотически сходятся к истинным апостериорным вероятностям.
>
> SVM, напротив, нацелен на оценку разделяющей поверхности, а не вероятностей. Его функция потерь (hinge) не является скор-функцией (strictly proper scoring rule), и выход $f(\mathbf{x})$ не имеет вероятностной шкалы. На практике применяют метод Platt scaling: подгоняют дополнительную сигмоиду $\hat{p} = \sigma(A f(\mathbf{x}) + B)$ на отложенной выборке. Однако такая калибровка может быть ненадёжной в областях с малой плотностью данных или при мультимодальных распределениях.
>
> KLR не требует постобработки и даёт хорошо откалиброванные вероятности во всей области, что особенно ценно в задачах, где важна не только точность классификации, но и качество вероятностных прогнозов (например, в медицине, финансах, рекомендательных системах). Платой служит отсутствие встроенного механизма разреженности и, как следствие, большие вычислительные затраты на этапе предсказания.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Почему ядерная логистическая регрессия считается ядровым методом? Какая теорема позволяет записать решение в виде $f(\mathbf{x}) = \sum_i \alpha_i K(\mathbf{x}, \mathbf{x}_i)$?
2. Как выглядит регуляризатор в задаче KLR и какую роль он играет? Запишите целевую функцию.
3. В чём отличие KLR от SVM с точки зрения получаемого решения (разреженность) и вида функции потерь?
4. Почему KLR выдаёт вероятности принадлежности классу, а SVM — нет? Как из KLR получить $p(y=+1 \mid \mathbf{x})$?
5. Чем функция потерь в KLR принципиально отличается от квадратичной потери в KRR?

**Для профессионалов:**

1. Выведите градиент и гессиан целевой функции KLR по $\boldsymbol{\alpha}$. Покажите, что гессиан представим в виде $\mathbf{K} \mathbf{W} \mathbf{K} + \lambda \mathbf{K}$, и объясните, почему он положительно определён.
2. Объясните, почему матрица $\mathbf{K}$ может быть плохо обусловлена, и как регуляризация $\lambda$ и метод IRLS помогают смягчить эту проблему. Какие ещё приёмы используются?
3. Сравните вычислительную сложность обучения KLR и KRR. Почему KLR требует итераций, а KRR даёт решение за один шаг?
4. Покажите связь KLR с гауссовскими процессами классификации: в каком смысле KLR является приближением к байесовскому выводу с логистической функцией связи?
5. Приведите пример задачи, где KLR демонстрирует значительно лучшую калибровку вероятностей, чем SVM с Platt scaling. Объясните, чем обусловлено преимущество.

### 5. Реализация ядерной логистической регрессии на Python

Теоретические основы ядерной логистической регрессии, изложенные в предыдущем разделе, естественно воплощаются в компактную программную реализацию. В этом разделе мы опишем структуру класса `KernelLogisticRegression`, алгоритм обучения с помощью итеративно перевзвешиваемого метода наименьших квадратов (IRLS), протестируем модель на синтетических и реальных данных, исследуем влияние гиперпараметров и обсудим вычислительные ограничения.

---

#### 5.1. Структура класса `KernelLogisticRegression`

Класс инкапсулирует ядровую функцию, параметры регуляризации и обученные коэффициенты. Основные атрибуты:

- `kernel` — строка или вызываемый объект, задающий ядро (`'rbf'`, `'poly'`, `'linear'` и т.д.).
- `gamma`, `degree`, `coef0` — параметры ядра.
- `lambda_` — коэффициент регуляризации ($\lambda > 0$).
- `alpha_` — вектор коэффициентов $\boldsymbol{\alpha} \in \mathbb{R}^n$, полученный после обучения.
- `X_train_` — сохранённые обучающие данные для вычисления ядер на этапе предсказания.

Метод `fit(X, y)` реализует обучение. На вход подаются матрица признаков `X` формы `(n_samples, n_features)` и вектор меток `y` со значениями $-1$ и $+1$. Основные шаги:

1. **Вычислить ядерную матрицу** $\mathbf{K}$ размера $n \times n$: $\mathbf{K}_{ij} = K(\mathbf{x}_i, \mathbf{x}_j)$. Для RBF-ядра $\mathbf{K} = \exp(-\gamma \cdot \text{pairwise\_sqdist})$.
2. **Инициализировать** $\boldsymbol{\alpha} = \mathbf{0}$ (или малыми случайными значениями).
3. **Организовать цикл IRLS**. На каждой итерации:
   - Вычислить линейную комбинацию $\mathbf{f} = \mathbf{K} \boldsymbol{\alpha}$.
   - Вычислить вероятности $p_i = \sigma(y_i f_i) = 1 / (1 + e^{-y_i f_i})$.
   - Вычислить веса $w_i = p_i (1 - p_i)$. Для численной устойчивости обрезать $w_i$ снизу (например, $10^{-8}$).
   - Вычислить рабочий отклик: $z_i = f_i - \frac{p_i - y_i}{w_i}$ для $i = 1,\dots,n$ (здесь учтено, что $y_i \in \{-1, 1\}$; при $y_i=+1$, $p_i \approx 1$ при больших $f_i$ и т.д.).
   - Решить взвешенную регуляризованную линейную систему:
     $$(\mathbf{K} \mathbf{W} \mathbf{K} + \lambda \mathbf{K}) \boldsymbol{\alpha}_{\text{new}} = \mathbf{K} \mathbf{W} \mathbf{z},$$
     где $\mathbf{W} = \operatorname{diag}(w_1,\dots,w_n)$. При $\lambda > 0$ матрица положительно определена. Систему можно упростить, умножив обе стороны на $\mathbf{K}^{-1}$ (если $\mathbf{K}$ обратима), что даёт
     $$(\mathbf{K} + \lambda \mathbf{W}^{-1}) \tilde{\boldsymbol{\alpha}} = \mathbf{z},$$
     где $\tilde{\boldsymbol{\alpha}} = \mathbf{K} \boldsymbol{\alpha}_{\text{new}}$. На практике удобнее решать систему относительно $\tilde{\boldsymbol{\alpha}}$ и затем положить $\boldsymbol{\alpha}_{\text{new}} = \mathbf{W} (\mathbf{K} \mathbf{W} + \lambda \mathbf{I})^{-1} \mathbf{z}$? Стандартный подход: решаем $(\mathbf{K} \mathbf{W} \mathbf{K} + \lambda \mathbf{K}) \boldsymbol{\alpha}_{\text{new}} = \mathbf{K} \mathbf{W} \mathbf{z}$. Используя матричное тождество, можно переписать как $\mathbf{K} (\mathbf{W} \mathbf{K} + \lambda \mathbf{I}) \boldsymbol{\alpha}_{\text{new}} = \mathbf{K} \mathbf{W} \mathbf{z}$. Если $\mathbf{K}$ невырождена, умножаем слева на $\mathbf{K}^{-1}$: $(\mathbf{W} \mathbf{K} + \lambda \mathbf{I}) \boldsymbol{\alpha}_{\text{new}} = \mathbf{W} \mathbf{z}$. Это $n \times n$ система, которую можно решать напрямую.
   - Обновить $\boldsymbol{\alpha} \leftarrow \boldsymbol{\alpha}_{\text{new}}$.
4. **Критерий остановки**: изменение нормы $\boldsymbol{\alpha}$ или логарифма правдоподобия меньше порога, либо достигнуто максимальное число итераций.
5. Сохранить финальное $\boldsymbol{\alpha}$ и `X_train` для последующих предсказаний.

Метод `predict_proba(X_new)`:
- Вычислить матрицу $\mathbf{K}_{\text{new}}$ размера `(n_new, n_train)`:
$(\mathbf{K}_{\text{new}})_{ij} = K(\mathbf{x}_{\text{new},i}, \mathbf{x}_j)$.
- Вычислить $f_{\text{new}} = \mathbf{K}_{\text{new}} \boldsymbol{\alpha}$.
- Вернуть вероятности $\hat{p}_i = \sigma(f_{\text{new},i})$.

Метод `predict(X_new)` возвращает метки $\hat{y}_i = \operatorname{sign}(f_{\text{new},i})$.

---

#### 5.2. Тестирование на синтетических данных: проблема XOR

Классическая проблема XOR — четыре точки на плоскости с координатами $(1,1), (1,-1), (-1,1), (-1,-1)$, где класс $+1$ присваивается, если знаки координат совпадают. Линейная логистическая регрессия (соответствующая линейному ядру) неспособна разделить такие данные, поскольку требуется нелинейное преобразование. Ядерная логистическая регрессия с RBF-ядром при правильно выбранных $\gamma$ и $\lambda$ строит гладкую разделяющую поверхность, идеально разделяющую XOR. Визуализация границы решения показывает нелинейные изгибы, соответствующие радиальным базисным функциям.

---

#### 5.3. Сравнение с полиномиальными признаками и sklearn

На реальных данных, например, на задаче Breast Cancer (признаки из sklearn), можно сравнить несколько подходов: линейная логистическая регрессия (LogisticRegression из sklearn), логистическая регрессия с полиномиальными признаками (PolynomialFeatures + LogisticRegression), и ядерная логистическая регрессия с RBF-ядром.

Метрики: ROC-AUC для оценки разделяющей способности; калибровочная кривая (reliability curve) для оценки качества вероятностей. Как правило, KLR с RBF-ядром при грамотной настройке достигает AUC, сравнимой с полиномиальным расширением, но превосходит по калибровке: её вероятности ближе к реальной частоте классов в каждом интервале уверенности. Это объясняется тем, что логарифмическая функция потерь напрямую оптимизирует кросс-энтропию, тогда как полиномиальная регрессия с высокой степенью может давать «слишком уверенные» прогнозы без специальной регуляризации.

---

#### 5.4. Влияние гиперпараметров $\lambda$ и $\gamma$

Регуляризация $\lambda$ и масштаб ядра $\gamma$ существенно влияют на форму решающей границы и качество модели.

- **$\lambda$** управляет балансом между подгонкой и гладкостью. При $\lambda \to 0$ модель способна идеально разделить обучающие данные, но рискует переобучиться, формируя сильно изрезанные границы. При $\lambda \to \infty$ коэффициенты $\boldsymbol{\alpha}$ стремятся к нулю, и модель вырождается в константную (вероятность $\approx 0.5$ для всех точек). Оптимальное $\lambda$ находят по сетке с кросс-валидацией, минимизируя log loss на отложенных фолдах.

- **$\gamma$** RBF-ядра задаёт типичный радиус влияния точки. Малое $\gamma$ (широкое ядро) даёт почти линейную границу, большое $\gamma$ (узкое ядро) делает модель локальной, что ведёт к переобучению. Визуализация кривых log loss на валидации при разных $\gamma$ демонстрирует U-образную форму с минимумом в зоне умеренных значений.

На практике часто проводят совместную оптимизацию $\lambda$ и $\gamma$ с помощью GridSearchCV или байесовской оптимизации.

---

#### 5.5. Ограничения и приближённые методы

Как и другие ядровые методы, KLR страдает от кубической сложности обучения ($O(n^3)$ на итерацию при прямом решении системы) и квадратичных затрат памяти на хранение матрицы $\mathbf{K}$. Для $n \lesssim 10^4$ это приемлемо, но при больших выборках требуются аппроксимации.

- **Nyström-приближение:** выбираются $m \ll n$ опорных точек, и матрица $\mathbf{K}$ аппроксимируется низкоранговым разложением. Это снижает сложность до $O(n m^2 + m^3)$ на итерацию.
- **Случайные признаки (Random Fourier Features):** для стационарных ядер генерируются $D$ случайных признаков, и задача сводится к линейной логистической регрессии в $D$-мерном пространстве, где можно применять стохастические градиентные методы и не хранить полную матрицу.
- **Итерационные решатели:** вместо прямого решения системы можно применять метод сопряжённых градиентов, используя быстрое умножение матрицы $\mathbf{K}$ на вектор (например, через fast multipole методы для RBF).

Кроме того, число итераций IRLS обычно невелико (5–20), а при использовании LBFGS можно отказаться от хранения гессиана, что ещё уменьшает затраты.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Как метод IRLS применяется к ядерной логистической регрессии? Какие шаги он включает?
2. Зачем в IRLS вводятся веса $w_i$ и как они вычисляются?
3. Как изменится формула обновления для многоклассового случая (softmax-функция)?
4. Почему алгоритм IRLS может не сходиться? Какие численные проблемы могут возникнуть?

**Для профессионалов:**

1. Выведите матричное уравнение для обновления $\boldsymbol{\alpha}$ в IRLS для KLR и покажите, как оно сводится к $(\mathbf{W} \mathbf{K} + \lambda \mathbf{I}) \boldsymbol{\alpha}_{\text{new}} = \mathbf{W} \mathbf{z}$ при невырожденной $\mathbf{K}$.
2. Реализуйте альтернативный оптимизатор на основе LBFGS, не требующий решения линейной системы. Сравните число итераций и устойчивость.
3. Добавьте в класс функциональность ранней остановки по валидационной выборке: как отслеживать log loss на отложенных данных и прекращать обучение при его возрастании?
4. Объясните, как адаптировать KLR для мультиклассовой классификации (multinomial). Какие изменения вносятся в функцию потерь и в метод IRLS?
5. Сравните скорость сходимости IRLS и обычного градиентного спуска при обучении KLR. В каких ситуациях градиентный спуск может оказаться предпочтительнее?

### 6. Сравнение Kernel PCA, KLR, KRR и SVM на реальных данных

В предыдущих разделах мы детально изучили четыре ядровых метода: Kernel PCA (снижение размерности и визуализация), Kernel Ridge Regression (регрессия), Kernel Logistic Regression (вероятностная классификация) и SVM (классификация с максимизацией зазора). Теперь пришло время столкнуть их на реальной задаче, чтобы увидеть сильные и слабые стороны каждого и выработать практические рекомендации. В качестве тестового полигона мы используем подвыборку рукописных цифр MNIST — игрушечную, но достаточно богатую для демонстрации нелинейных эффектов.

---

#### 6.1. Визуализация MNIST с помощью Kernel PCA

MNIST содержит 70 000 изображений цифр $28 \times 28$ (784 пикселя). Для наглядности возьмём подмножество из 1000 случайных точек, охватывающее все 10 классов, и применим два метода снижения размерности до двух компонент.

- **Линейный PCA** просто ищет направления максимальной дисперсии в исходном 784-мерном пространстве. Двумерная проекция сохраняет глобальную структуру, но классы заметно перемешаны: цифры разных типов образуют перекрывающиеся облака, особенно схожие по начертанию (например, 4 и 9). Объяснённая дисперсия первых двух компонент невелика (около 15–20%), и визуализация не выявляет чёткой кластерной структуры.

- **Kernel PCA с RBF-ядром** ($\gamma \approx 10^{-3} \dots 10^{-2}$, подобранным по разбросу данных) даёт существенно более выразительное вложение. Благодаря нелинейному отображению в RKHS, классы становятся компактнее и разделяются заметно лучше. Цифры «0», «1», «7» часто образуют изолированные группы; сложные цифры (3, 5, 8) демонстрируют более чёткую внутреннюю структуру. Проекция на первые две ядерные главные компоненты визуально приближается к тому, что дают нелинейные методы типа t-SNE, но, в отличие от них, Kernel PCA имеет явную формулу преобразования и может применяться к новым точкам без переобучения.

Вывод: Kernel PCA с правильно настроенным ядром способен улавливать нелинейные многообразия, что делает его мощным инструментом визуализации высокоразмерных данных.

---

#### 6.2. Классификация: KLR против SVM

Для бинарной классификации выделим цифры 0 и 1 (наиболее легко разделимые). Обучим на 500 точках и протестируем на 200. Применим KLR и SVM, оба с RBF-ядром.

- **SVM** быстро обучается (особенно с библиотекой LIBSVM/SVC) и даёт разреженное решение: из 500 обучающих точек опорными становятся лишь несколько десятков. Точность классификации на тесте близка к 100%, граница решения гладкая, но сами значения решающей функции не откалиброваны: предсказание лежит в диапазоне, не соответствующем вероятности.

- **KLR** оптимизирует логарифмическую потерю, что требует решения плотной задачи — в пределе все $\alpha_i$ отличны от нуля. Обучение IRLS сходится за 10–15 итераций, точность также около 100%, но предсказанные вероятности $p(y=+1 \mid \mathbf{x})$ хорошо откалиброваны: в областях, где классы смешаны (например, вблизи границы), вероятности плавно меняются от 0 до 1, в то время как SVM выдаёт значения, не имеющие вероятностного смысла. Время обучения KLR при $n=500$ пренебрежимо мало, но уже при $n=5000$ кубическая сложность даёт о себе знать: SVM обучается быстрее из-за разреженности и эффективных солверов, KLR требует хранения и обращения полной матрицы $\mathbf{K}$.

Таким образом, при малых и средних выборках KLR предпочтителен, когда важна вероятностная интерпретация. SVM выигрывает при больших $n$ и когда достаточно лишь метки класса.

---

#### 6.3. Конвейер «снижение размерности + классификация»

Часто на практике применяют двухэтапный подход: сначала Kernel PCA сокращает размерность, затем классификатор (линейный или ядерный) обучается на полученных проекциях. Протестируем это на полном наборе из 10 классов MNIST (уже не бинарном). Для многоклассовой классификации будем использовать One-vs-Rest SVM и мультиномиальную логистическую регрессию с ядром (приближённую через случайные признаки).

1. **Без снижения размерности:** SVM с RBF-ядром на 784-мерных данных даёт высокое качество ($>98\%$), но обучение требует тщательной настройки $\gamma$ и $C$.
2. **Kernel PCA (50 компонент) + линейная логистическая регрессия:** проект на первые 50 ядерных главных компонент и затем применяем мультиклассовую логистическую регрессию. Качество может оказаться сравнимым с прямым SVM, но обучение значительно быстрее (особенно предсказание, так как компоненты предвычислены). Более того, 50-мерное представление можно использовать повторно для разных классификаторов.
3. **Прямой KLR с RBF** (в бинарном случае) и KLR на проекциях: прямое KLR даёт максимальную гибкость, но ценой плотного представления; KLR на проекциях проигрывает в точности лишь доли процента, но выигрывает в скорости.

Это демонстрирует, что Kernel PCA может служить не только для визуализации, но и как этап feature extraction, ускоряющий последующие алгоритмы за счёт работы в низкоразмерном пространстве, сохраняющем нелинейную структуру.

---

#### 6.4. Выбор гиперпараметров кросс-валидацией

Общая стратегия для всех ядровых методов: сеточный поиск по $\gamma$ (для RBF) и параметру регуляризации ($\lambda$ в KRR/KLR, $C$ в SVM) с использованием $k$-фолд кросс-валидации.

- **KRR, KLR:** минимизируем log loss или RMSE на отложенных фолдах. Для KLR удобно отслеживать сходимость и останавливаться при росте валидационной ошибки (ранняя остановка).
- **SVM:** как правило, оптимизируют accuracy или F1-score; $C$ и $\gamma$ ищут на логарифмической сетке.
- **Kernel PCA:** число компонент $d$ выбирают по накопленной объяснённой дисперсии в $\mathcal{H}$ (сумма $\lambda_k / \sum \lambda_k$) или по downstream-задаче (например, точности классификации на проекциях). $\gamma$ также подбирают кросс-валидацией, иногда визуальным анализом кластеризации.

При реализации удобно использовать `GridSearchCV` из sklearn, оборачивая наши собственные классы KLR, KRR в интерфейс, совместимый с sklearn.

---

#### 6.5. Выводы: когда какой метод предпочтительнее

Сведём полученные наблюдения в практические рекомендации.

| Метод          | Основное применение                         | Вероятности | Разреженность | Масштабируемость      |
|----------------|---------------------------------------------|-------------|---------------|-----------------------|
| **KRR**        | Нелинейная регрессия, гладкие функции        | Нет         | Плотное       | $O(n^3)$, аппроксимации |
| **KLR**        | Классификация с калиброванными вероятностями | Да          | Плотное       | $O(n^3)$, IRLS/Newton  |
| **SVM**        | Классификация с максимизацией зазора         | Нет (Platt) | Разреженное   | $O(n^2)$–$O(n^3)$      |
| **Kernel PCA** | Визуализация, денузинг, feature extraction    | —           | Плотное       | $O(n^3)$, Nyström      |

**Когда выбирать каждый из них:**

- *Kernel PCA*: если нужно исследовать нелинейную структуру данных, визуализировать многомерные облака, очистить данные от шума или подготовить компактное нелинейное представление для других методов.
- *KRR*: когда решается задача регрессии, отклик непрерывен, а связь с признаками предположительно нелинейна; когда важна аналитическая форма решения и его байесовская интерпретация (гауссовские процессы).
- *KLR*: если нужны именно вероятности принадлежности классам (оценка рисков, кредитный скоринг), и размер выборки не превышает нескольких десятков тысяч. Даёт хорошо откалиброванные прогнозы.
- *SVM*: когда первостепенна точность разделения классов, выборка велика, и вероятностный выход не обязателен. Разреженность SVM ускоряет предсказание и снижает потребление памяти.

В реальных проектах часто комбинируют методы: например, Kernel PCA + KLR/SVM, или используют несколько ядер с Multiple Kernel Learning. Универсального «лучшего» метода нет — выбор диктуется природой данных, требуемой интерпретируемостью и вычислительными ресурсами.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Какую роль играет Kernel PCA в конвейере анализа данных? Приведите пример, когда он полезен перед классификацией.
2. Почему снижение размерности с помощью Kernel PCA может улучшить качество классификации? В каких случаях оно не помогает?
3. Как выбрать число главных компонент в Kernel PCA? Какие критерии вы знаете?
4. Сравните время обучения KLR и SVM при $n = 5000$. Почему SVM обычно быстрее?
5. Что такое «предвычисленное ядро» и как оно помогает ускорить подбор гиперпараметров?

**Для профессионалов:**

1. Проведите анализ чувствительности Kernel PCA к выбросам: как наличие нескольких аномальных точек влияет на центрирование и главные компоненты? Предложите робастную модификацию.
2. Сравните Kernel PCA с автокодировщиками (нейронными сетями) для нелинейного снижения размерности. В каких условиях каждый из них предпочтительнее?
3. Предложите метод ускорения обучения KLR через аппроксимацию ядерной матрицы (например, Nyström) и опишите, как изменится уравнение IRLS.
4. Объясните, как ядерная логистическая регрессия справляется с несбалансированными классами. Достаточно ли ввести веса примеров в функцию потерь, и как это повлияет на IRLS?
5. Реализуйте KLR с использованием случайных признаков (Random Fourier Features) для RBF-ядра. Выпишите целевую функцию и градиентный шаг для стохастического спуска.